<a href="https://colab.research.google.com/github/nawroz-m/ML_learning/blob/healthcare/Egg_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [103]:
from google.colab import drive
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split

In [104]:
drive.flush_and_unmount()
drive.mount("/content/drive")

Mounted at /content/drive


In [105]:
# read the dataset
path = "/content/drive/MyDrive/Healthcare/EGG/EEG-data.xlsx"
df = pd.read_excel(path)

In [106]:
# get the label
label = df.iloc[:, -1]
# get collumns where the last element is 2 & 3
c2 = df[label==2].iloc[:, :-1]
c3 = df[label==3].iloc[:, :-1]

In [107]:
# prepare the features and atribute dataste
X=np.vstack((c2, c3))
y=np.hstack((np.zeros(len(c2)), np.ones(len(c3))))

In [108]:
# prepare training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=None, shuffle=True)

**Decision Tree Classifier**

In [109]:
# Create a simple decision tree classifier to classify c2 from c3 model
decision_tree = sklearn.tree.DecisionTreeClassifier(criterion='gini', max_depth=20, random_state=None)

In [110]:
# train the decision tree classifier
decision_tree.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=20)

In [111]:
# predict the decision tree classifier on test dataset
y_pred = decision_tree.predict(X_test)

# get the accuracy of decision tree prediction
accuracy = np.sum(y_pred == y_test)/len(y_test)
accuracy

np.float64(0.525)

**Random Forest Classifier**

In [112]:
# Ceate a random forest classifier
rfc_model = sklearn.ensemble.RandomForestClassifier(n_estimators=300,
                                                    criterion='gini',
                                                    max_depth=80,
                                                    random_state=None,
                                                    max_features='log2')

In [113]:
# train the random forest classifier
rfc_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=80, max_features='log2', n_estimators=300)

In [115]:
# Make a prediction to see the random forrest accuracy score
rfc_pred = rfc_model.predict(X_test)
# get the accuracy
rfc_accuracy = np.sum(rfc_pred == y_test)/len(y_test)
rfc_accuracy

np.float64(0.825)

**Grid Search with Random Forrest**

In [118]:
# create a new model
rfc_grid_model = sklearn.ensemble.RandomForestClassifier(n_estimators=300,
                                                         criterion='gini',
                                                         max_depth=80,
                                                         random_state=None,
                                                         max_features='log2')

In [133]:
# train with grid search
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [20, 40, 80],
    "min_samples_split": [5, 10],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "criterion": ["gini", "entropy"]
}
# create grid search model
grid = sklearn.model_selection.GridSearchCV(rfc_grid_model,
                                            param_grid,
                                              cv=3)
# train the grid search
grid.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=RandomForestClassifier(max_depth=80, max_features='log2',
                                              n_estimators=300),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [20, 40, 80],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [5, 10],
                         'n_estimators': [100, 200, 300]})

In [136]:
# fid the best model
grid_search_best_model = grid.best_estimator_
# predict grid search best model on testing dataset
grid_y_pred = grid_search_best_model.predict(X_test)
# calcualte the score
grid_score = np.sum(grid_y_pred == y_test)/len(y_test)
grid_score

np.float64(0.85)

**Random Search with Random Forest**

In [137]:
# create a new model
rfc_rs_model = sklearn.ensemble.RandomForestClassifier(n_estimators=300,
                                                         criterion='gini',
                                                         max_depth=80,
                                                         random_state=None,
                                                         max_features='log2')
random_search_model = sklearn.model_selection.RandomizedSearchCV(rfc_rs_model, param_grid, cv=3)

In [138]:
random_search_model.fit(X_train, y_train)

RandomizedSearchCV(cv=3,
                   estimator=RandomForestClassifier(max_depth=80,
                                                    max_features='log2',
                                                    n_estimators=300),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [20, 40, 80],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2],
                                        'min_samples_split': [5, 10],
                                        'n_estimators': [100, 200, 300]})

In [139]:
random_search_best_model = random_search_model.best_estimator_

In [140]:
# make some prediction
random_search_y_pred = random_search_best_model.predict(X_test)
# the accuracy score
random_search_score = np.sum(random_search_y_pred == y_test)/len(y_test)
random_search_score

np.float64(0.775)